# Movie Recommendation System
### Content-Based Filtering — TMDB API
---

In [ ]:
import numpy as np
import pandas as pd
import requests
import pickle
import time
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from nltk.stem.porter import PorterStemmer
import nltk
nltk.download('punkt', quiet=True)

In [ ]:
API_KEY  = 'ebef1aa0b639138c3040e6929ea9f1eb'
BASE_URL = 'https://api.themoviedb.org/3'
IMG_BASE = 'https://image.tmdb.org/t/p/w500'

## Step 1 — Fetch Movie List from TMDB

In [ ]:
def get_movie_list(endpoint, pages=50):
    movies = []
    for page in range(1, pages + 1):
        r = requests.get(
            f'{BASE_URL}/movie/{endpoint}',
            params={'api_key': API_KEY, 'language': 'en-US', 'page': page}
        )
        movies.extend(r.json().get('results', []))
        time.sleep(0.05)
    return movies

popular     = get_movie_list('popular',      pages=50)
top_rated   = get_movie_list('top_rated',    pages=50)
now_playing = get_movie_list('now_playing',  pages=20)   # latest theatrical releases

all_movies = {m['id']: m for m in popular + top_rated + now_playing}
print(f'Total unique movies: {len(all_movies)}')

## Step 2 — Fetch Details, Credits & Keywords

In [ ]:
def fetch_movie_data(movie_id):
    params = {'api_key': API_KEY, 'language': 'en-US'}
    detail   = requests.get(f'{BASE_URL}/movie/{movie_id}',          params=params).json()
    credits  = requests.get(f'{BASE_URL}/movie/{movie_id}/credits',  params=params).json()
    keywords = requests.get(f'{BASE_URL}/movie/{movie_id}/keywords', params=params).json()
    return detail, credits, keywords

records = []

for i, (movie_id, m) in enumerate(all_movies.items()):
    try:
        detail, credits, kw = fetch_movie_data(movie_id)
        genres   = [g['name'] for g in detail.get('genres', [])]
        keywords = [k['name'] for k in kw.get('keywords', [])]
        cast     = [c['name'] for c in credits.get('cast', [])[:3]]
        director = [c['name'] for c in credits.get('crew', []) if c.get('job') == 'Director'][:1]
        records.append({
            'movie_id':     movie_id,
            'title':        detail.get('title', ''),
            'overview':     detail.get('overview', ''),
            'poster_path':  detail.get('poster_path', ''),
            'vote_average': detail.get('vote_average', 0),
            'genres':       genres,
            'keywords':     keywords,
            'cast':         cast,
            'crew':         director,
        })
        time.sleep(0.05)
        if (i + 1) % 100 == 0:
            print(f'{i+1}/{len(all_movies)} done...')
    except Exception as e:
        print(f'Skipped {movie_id}: {e}')

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
df.head(3)

## Step 3 — Clean Data

In [ ]:
df = df[df['overview'].str.strip() != ''].dropna(subset=['overview']).reset_index(drop=True)
print(f'After cleaning: {df.shape}')
df.isnull().sum()

## Step 4 — Build Tags

In [ ]:
for col in ['genres', 'keywords', 'cast', 'crew']:
    df[col] = df[col].apply(lambda lst: [x.replace(' ', '') for x in lst])

# Feature weighting: repeat high-signal fields so TF-IDF scores them higher
# genres × 3, director × 3, cast × 2, keywords × 2, overview × 1
df['tags'] = (
    df['overview'].apply(str.split) +
    df['genres']   * 3 +
    df['crew']     * 3 +
    df['cast']     * 2 +
    df['keywords'] * 2
)
df['tags'] = df['tags'].apply(lambda x: ' '.join(x).lower())
df[['title', 'tags']].head(3)

## Step 5 — Stemming

In [ ]:
ps = PorterStemmer()
df['tags'] = df['tags'].apply(lambda text: ' '.join(ps.stem(w) for w in text.split()))
print(df.loc[0, 'tags'][:300])

## Step 6 — Vectorize & Cosine Similarity

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors    = cv.fit_transform(df['tags']).toarray()
similarity = cosine_similarity(vectors)
print(f'Vectors:    {vectors.shape}')
print(f'Similarity: {similarity.shape}')

## Step 7 — Test Recommendation

In [ ]:
def recommend(title, n=5):
    match = df[df['title'].str.lower() == title.lower()]
    if match.empty:
        print(f"'{title}' not found.")
        return
    idx   = match.index[0]
    top_n = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    print(f'Recommendations for "{title}":')
    for rank, (i, score) in enumerate(top_n, 1):
        print(f'  {rank}. {df.iloc[i].title}  ({score:.3f})')

recommend('Avatar')

## Step 8 — Save Model

In [ ]:
movies_export = df[['movie_id', 'title', 'overview', 'poster_path', 'vote_average']].copy()
pickle.dump(movies_export.to_dict(), open('movies_dict.pkl', 'wb'))
pickle.dump(similarity,              open('similarity.pkl',  'wb'))
print(f'Saved movies_dict.pkl  ({len(movies_export)} movies)')
print('Saved similarity.pkl')